In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
df = pd.read_csv("SMS.csv",encoding = "latin1")
df.head()

,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [4]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Define these OUTSIDE the function to save processing time
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()

def clean_text(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs FIRST (while special characters like http:// exist)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # 4. Tokenization
    words = word_tokenize(text)
    
    # 5. Remove Stopwords and Stemming in one pass
    # This is more efficient than doing two separate loops
    cleaned_words = [STEMMER.stem(w) for w in words if w not in STOP_WORDS]

    return " ".join(cleaned_words)

# Apply to your dataframe
df["clean_text"] = df["sms"].apply(clean_text)

In [5]:
import nltk
nltk.download('punkt_tab')
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Sandeep\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sandeep\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
df["label"].value_counts()

label
0    4827
1     747
Name: count, dtype: int64

In [7]:
from imblearn.over_sampling import RandomOverSampler

x = df.drop("label",axis=1)
y = df["label"]

os = RandomOverSampler()
x_res,y_res = os.fit_resample(x,y)

df_bal = pd.DataFrame(x_res)
df_bal["label"] = y_res

df_bal['label'].value_counts()

label
0    4827
1    4827
Name: count, dtype: int64

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

x_train,x_test,y_train,y_test=train_test_split(df["clean_text"],df["label"],test_size=0.2, random_state=42)
tfid =TfidfVectorizer()

x_train_tfid = tfid.fit_transform(x_train)
y_test_tfid= tfid.transform(x_test)
x_test_tfid = tfid.transform(x_test)


x_train_tfid.toarray()


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
rfc = RandomForestClassifier()

rfc.fit(x_train_tfid,y_train)
y_pred = rfc.predict(x_test_tfid)

cm = confusion_matrix(y_test,y_pred)
cr = classification_report(y_test,y_pred)

print("confusion matrix ",cm)
print("classification_report",cr)




confusion matrix  [[954   0]
 [ 29 132]]
classification_report               precision    recall  f1-score   support

           0       0.97      1.00      0.99       954
           1       1.00      0.82      0.90       161

    accuracy                           0.97      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [27]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()

lr.fit(x_train_tfid,y_train)
y_pred = lr.predict(x_test_tfid)

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test,y_pred)
cr = classification_report(y_test,y_pred)

print("confusion matrix ",cm)
print("classification_report",cr)



confusion matrix  [[950   4]
 [ 38 123]]
classification_report               precision    recall  f1-score   support

           0       0.96      1.00      0.98       954
           1       0.97      0.76      0.85       161

    accuracy                           0.96      1115
   macro avg       0.97      0.88      0.92      1115
weighted avg       0.96      0.96      0.96      1115



In [29]:
from sklearn.svm import SVC
svc =SVC()

svc.fit(x_train_tfid,y_train)
y_pred = svc.predict(x_test_tfid)

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test,y_pred)
cr = classification_report(y_test,y_pred)

print("confusion matrix ",cm)
print("classification_report",cr)

confusion matrix  [[951   3]
 [ 20 141]]
classification_report               precision    recall  f1-score   support

           0       0.98      1.00      0.99       954
           1       0.98      0.88      0.92       161

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [32]:
from sklearn.ensemble import VotingClassifier

ens_cl =VotingClassifier( estimators = [
    ('random_forest',rfc),
    ('Logistic_regression', lr),
    (" SVC", svc)
],voting ="hard")


ens_cl.fit(x_train_tfid,y_train)
y_pred = ens_cl.predict(x_test_tfid)

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test,y_pred)
cr = classification_report(y_test,y_pred)

print("confusion matrix ",cm)
print("classification_report",cr)


confusion matrix  [[952   2]
 [ 24 137]]
classification_report               precision    recall  f1-score   support

           0       0.98      1.00      0.99       954
           1       0.99      0.85      0.91       161

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [38]:
import pickle

pickle.dump(rfc,open("model.pkl","wb"))
pickle.dump(tfid,open("tfid_vect.pkl","wb"))

with open("model.pkl","rb") as file:
    rfc =pickle.load(file)

with open("tfid_vect.pkl","rb") as file:
    tfid =pickle.load(file)

In [42]:
def detection(text):
    cleaned_text = clean_text(text)
    vector_text = tfid.transform([cleaned_text])
    result = rfc.predict(vector_text)
    return [0]
    

In [45]:
input_text=(" hi rahul today meeting is on 9:00 am i hope you come on time ")
prediction = detection(input_text)

if prediction == 0:
    print(" real msg ")
else:
    print( " Spam msg ")

 Spam msg 
